In [ ]:
l=glob("Datasets/1.HAM10000/siamese/*/*/*")
random.shuffle(l)

Original=cv2.imread(l[0], cv2.IMREAD_UNCHANGED)
# Original=cv2.cvtColor(Original, cv2.COLOR_BGR2RGB)

# 1. Rotation: Rotating the image by a specific angle.
Rotate=A.Compose([A.augmentations.geometric.transforms.Affine(rotate=(-45,45), mode=2, p=1)], p=1)
Rotate_img=Rotate(image=Original)['image']

# 2. Translation: Shifting the image along the x or y axis.
Translate=A.Compose([A.augmentations.geometric.transforms.Affine(translate_percent=(-0.1,0.1), mode=2, p=1)], p=1)
Translate_img=Translate(image=Original)['image']

# 3. Scaling/Resizing: Zooming in or out on the image.
Scale=A.Compose([A.augmentations.geometric.transforms.Affine(scale=(0.7,1.3), mode=2, p=1)], p=1)
Scale_img=Scale(image=Original)['image']

# 4. ShiftScaleRotate
ShiftScaleRotate=A.Compose([A.ShiftScaleRotate(p=1)], p=1)
ShiftScaleRotate_img=ShiftScaleRotate(image=Original)['image']

# 5. Horizontal Flipping:  
HorizontalFlip=A.Compose([A.HorizontalFlip(p=1)], p=1)
HorizontalFlip_img=HorizontalFlip(image=Original)['image']

# 6. Vertical Flipping:
VerticalFlip=A.Compose([A.VerticalFlip(p=1)], p=1)         
VerticalFlip_img=VerticalFlip(image=Original)['image']

# 7. ShearX: Applying a shear mapping to the image.
ShearX=A.Compose([A.augmentations.geometric.transforms.Affine(shear = {'x': (-20,20)}, mode=2, p=1)], p=1)
ShearX_img=ShearX(image=Original)['image']

# 8. ShearY:
ShearY=A.Compose([A.augmentations.geometric.transforms.Affine(shear = {'y': (-20,20)}, mode=2, p=1)], p=1)
ShearY_img=ShearY(image=Original)['image']

# 9. Color Jittering: Randomly varying brightness, contrast, saturation, and hue.
ColorJitter=A.Compose([A.ColorJitter(brightness=(0.8,1.2),contrast=(0.8,1.2),saturation=(0.8,1.2),hue=(-0.5,0.5),p=1)],p=1)
ColorJitter_img=ColorJitter(image=Original)['image'] 

# 10. Brightness Adjustment: Increasing or decreasing brightness, Contrast Adjustment: Enhancing or reducing the contrast.
RandomBrightnessContrast=A.Compose([A.RandomBrightnessContrast(p=1)], p=1)
RandomBrightnessContrast_img=RandomBrightnessContrast(image=Original)['image']

# 11. Saturation Adjustment: Modifying the saturation of colors. Hue Adjustment: Changing the hue to alter color balance.
HueSaturationValue=A.Compose([A.HueSaturationValue(p=1)], p=1)
HueSaturationValue_img=HueSaturationValue(image=Original)["image"]  

# 12. Sharpening: Enhancing edge details.
Sharpen=A.Compose([A.Sharpen(p=1)], p=1)    
Sharpen_img=Sharpen(image=Original)['image']

# 13. Gamma Correction: 
Gamma=A.Compose([A.RandomGamma(p=1)], p=1)
Gamma_img=Gamma(image=Original)["image"] 

# 14. Image Invert:
InvertImg=A.Compose([A.InvertImg(p=1)], p=1)
Invert_img=InvertImg(image=Original)['image']

# 15. Posterize
Posterize=A.Compose([A.Posterize(p=1)], p=1)
Posterize_img=Posterize(image=Original)['image']

# 16. Solarize
Solarize=A.Compose([A.Solarize(p=1)], p=1)
Solarize_img=Solarize(image=Original)['image']

# 17. AutoContrast
def autocontrast(image, cutoff=3):
    if len(image.shape) == 3:  # RGB image
        channels = cv2.split(image)
        auto_channels = [autocontrast_channel(ch, cutoff) for ch in channels]
        return cv2.merge(auto_channels)
    else:  # Grayscale image
        return autocontrast_channel(image, cutoff)

def autocontrast_channel(channel, cutoff):
    # Calculate percentiles
    low_val = np.percentile(channel, cutoff)
    high_val = np.percentile(channel, 100 - cutoff)

    # Stretch pixel values to [0, 255] range
    if high_val > low_val:
        scaled = (channel - low_val) * (255.0 / (high_val - low_val))
        scaled = np.clip(scaled, 0, 255).astype(np.uint8)
    else:
        scaled = channel.copy()

    return scaled

# Apply AutoContrast with a 3% cutoff
AutoContrast_img = autocontrast(Original, cutoff=3)

# 18. ChromaticAberration
ChromaticAberration=A.Compose([A.ChromaticAberration(mode='random',p=1)], p=1)
ChromaticAberration_img=ChromaticAberration(image=Original)["image"]

# 19. RGBShift
RGBShift=A.Compose([A.RGBShift(p=1)], p=1)
RGBShift_img=RGBShift(image=Original)["image"]

# 20. ToGray
ToGray=A.Compose([A.ToGray(method='weighted_average', p=1)], p=1) # weighted_average,from_lab,desaturation,average,max,pca
ToGray_img=ToGray(image=Original)["image"]

# 21. ToSepia
ToSepia=A.Compose([A.ToSepia(p=1)], p=1)
ToSepia_img=ToSepia(image=Original)["image"]

# 22. Gaussian Noise: Adding random pixel values following a Gaussian distribution.
GaussNoise=A.Compose([A.GaussNoise(p=1)], p=1)
GaussNoise_img=GaussNoise(image=Original)["image"]

# 23. Salt-and-Pepper Noise: Randomly setting pixels to either black or white.
class SaltAndPepperNoise(A.ImageOnlyTransform):
    def __init__(self,salt_prob=0.01,pepper_prob=0.01,always_apply=True,p=1.0):
        super(SaltAndPepperNoise, self).__init__(always_apply, p)
        self.salt_prob = salt_prob
        self.pepper_prob = pepper_prob

    def apply(self,img,**params):
        noisy_img = img.copy()
        salt_mask = np.random.rand(*img.shape[:2]) < self.salt_prob
        noisy_img[salt_mask] = 255
        pepper_mask = np.random.rand(*img.shape[:2]) < self.pepper_prob
        noisy_img[pepper_mask] = 0
        return noisy_img

salt_and_pepper=A.Compose([SaltAndPepperNoise(salt_prob=0.01,pepper_prob=0.01,p=1.0)])
SaltandPepper_img=salt_and_pepper(image=Original)["image"]

# 24. Speckle Noise: Multiplicative noise added to pixel intensities.
class SpeckleNoise(A.ImageOnlyTransform):
    def __init__(self,noise_prob=0.1,always_apply=True,p=1.0):
        super(SpeckleNoise, self).__init__(always_apply,p)
        self.noise_prob = noise_prob

    def apply(self, img, **params):
        noise = np.random.normal(0, self.noise_prob, img.shape)
        noisy_img = img + img * noise
        noisy_img = np.clip(noisy_img, 0, 255).astype(np.uint8)
        return noisy_img
        
speckle_noise=A.Compose([SpeckleNoise(noise_prob=0.1,p=1.0)])
Speckle_img=speckle_noise(image=Original)["image"]

# 25. JPEG Compression Artifacts: Simulating compression-induced noise.
ImageCompression=A.Compose([A.ImageCompression(quality_lower=10,quality_upper=30,p=1)],p=1)
JPEGCompression_img=ImageCompression(image=Original)["image"]

# 26. Blur:
Blur=A.Compose([A.Blur(p=1)], p=1)
BoxBlur_img=Blur(image=Original)["image"]

# 27. Gaussian Blur: Applying a Gaussian kernel to blur the image.
GaussianBlur=A.Compose([A.GaussianBlur(p=1)], p=1)
GaussianBlur_img=GaussianBlur(image=Original)["image"]

# 28. Motion Blur: Simulating motion-induced blurring.   
MotionBlur=A.Compose([A.MotionBlur(p=1)], p=1)
MotionBlur_img=MotionBlur(image=Original)["image"]

# 29. Median Blur:
MedianBlur=A.Compose([A.MedianBlur(p=1)], p=1)
MedianBlur_img=MedianBlur(image=Original)["image"]

# 30. Glass Blur:
GlassBlur=A.Compose([A.GlassBlur(p=1)], p=1)
GlassBlur_img=GlassBlur(image=Original)["image"]

# 31. Zoom Blur:
ZoomBlur=A.Compose([A.ZoomBlur(p=1)], p=1)
ZoomBlur_img=ZoomBlur(image=Original)["image"]

# 32. Defocus Blur:
Defocus=A.Compose([A.Defocus(radius=(3,5),p=1)], p=1)
Defocus_img=Defocus(image=Original)["image"]

# 33. Emboss
class EmbossEffect(A.ImageOnlyTransform):
    def __init__(self, emboss_strength=1.0, always_apply=False, p=1.0):
        super(EmbossEffect, self).__init__(always_apply, p)
        self.emboss_strength = emboss_strength

    def apply(self, img, **params):
        kernel = np.array([[-2,-1,0],[-1,1,1],[0,1,2]],dtype=np.float32)
        embossed_img = cv2.filter2D(img, -1, kernel * self.emboss_strength)
        embossed_img = np.clip(embossed_img, 0, 255).astype(np.uint8)
        return embossed_img

emboss=A.Compose([EmbossEffect(emboss_strength=1.0,p=1.0)])
Emboss_img=emboss(image=Original)["image"]

# 34. Channel Shuffling: Randomly rearranging RGB channels.
ChannelShuffle=A.Compose([A.ChannelShuffle(p=1)], p=1)
ChannelShuffle_img=ChannelShuffle(image=Original)["image"]

# 35. CLAHE
CLAHE=A.Compose([A.CLAHE(clip_limit=2.0, p=1)], p=1)
CLAHE_img=CLAHE(image=Original)['image']

# 36. Histogram Equalization: Enhancing contrast by equalizing the histogram.
Equalize=A.Compose([A.Equalize(p=1)], p=1)
Equalize_img=Equalize(image=Original)['image']

# 37. Histogram Equalization YUV: Enhancing contrast by equalizing the histogram.
def equalize_yuv(image):
    yuv_image = cv2.cvtColor(image, cv2.COLOR_BGR2YUV)
    yuv_image[:,:,0]=cv2.equalizeHist(yuv_image[:,:,0])
    equalized_image = cv2.cvtColor(yuv_image, cv2.COLOR_YUV2BGR)
    return equalized_image

EqualizeYUV_img=equalize_yuv(Original)

# 38. Elastic Transformations: Stretching or compressing areas in an elastic manner.
ElasticTransform=A.Compose([A.ElasticTransform(p=1)],p=1)
ElasticTransform_img=ElasticTransform(image=Original)['image']

# 39. Optical Distortion
OpticalDistortion=A.Compose([A.OpticalDistortion(distort_limit=0.5,p=1)],p=1)
OpticalDistortion_img=OpticalDistortion(image=Original)['image']

# 40. Perspective Transformations: Simulating changes in camera angle.
Perspective=A.Compose([A.Perspective(scale=0.2,p=1)],p=1)
Perspective_img=Perspective(image=Original)['image']

# 41. RandomGridShuffle: Rearranging patches within an image.
RandomGridShuffle=A.Compose([A.RandomGridShuffle(p=1)], p=1)
RandomGridShuffle_img=RandomGridShuffle(image=Original)["image"]

# 42. CoarseDropout: Masking random square regions.
CoarseDropout=A.Compose([A.CoarseDropout(num_holes_range=(20,40),hole_height_range=(20,40),
                                         hole_width_range=(20,40),p=1)],p=1)  # fill="inpaint_ns",
CoarseDropout_img=CoarseDropout(image=Original)["image"]

# 43. Dilation
dilation=A.Compose([A.Morphological(scale=(3, 5), operation='dilation', p=1)], p=1)
Dilation_img=dilation(image=Original)["image"]

# 44. Erosion
erosion=A.Compose([A.Morphological(scale=(3, 5), operation='erosion', p=1)], p=1)
Erosion_img=erosion(image=Original)["image"]

# 45. Vignetting: Adding a vignette effect to the image.
def apply_vignette(input_image, **kwargs):  
    rows, cols = input_image.shape[:2]
    intensity = 0.6
    X_resultant_kernel = cv2.getGaussianKernel(cols, cols * intensity)
    Y_resultant_kernel = cv2.getGaussianKernel(rows, rows * intensity)
    resultant_kernel = Y_resultant_kernel * X_resultant_kernel.T
    mask = 255 * resultant_kernel / np.linalg.norm(resultant_kernel)
    output = np.copy(input_image)
    for i in range(3):
        output[:,:,i] = output[:,:,i] * mask
    return output

vignette_trans=A.Compose([A.Lambda(image=apply_vignette, p=1)], p=1)
Vignette_img=vignette_trans(image=Original)["image"]

# 46. Shadowing: Simulating shadows on objects.
RandomShadow=A.Compose([A.RandomShadow(num_shadows_lower=2,num_shadows_upper=3,shadow_dimension=5,shadow_roi=(0,0,1,1),p=1)],p=1)
RandomShadow_img=RandomShadow(image=Original)["image"]

# 47. Adding rain 
RandomRain=A.Compose([A.RandomRain(brightness_coefficient=0.9,drop_width=1,blur_value=3,p=1)],p=1)
RandomRain_img=RandomRain(image=Original)["image"]

# 48. Adding snow 
RandomSnow=A.Compose([A.RandomSnow(brightness_coeff=2,snow_point_lower=0.4,snow_point_upper=0.7,p=1)],p=1)
RandomSnow_img=RandomSnow(image=Original)["image"]

# 49. Adding sun flare 
RandomSunFlare=A.Compose([A.RandomSunFlare(flare_roi=(0,0,1,1),src_radius=150,angle_lower=0.8,p=1)],p=1)
RandomSunFlare_img=RandomSunFlare(image=Original)["image"]    

# 50. Adding fog 
RandomFog=A.Compose([A.RandomFog(fog_coef_lower=0.5,fog_coef_upper=0.8,alpha_coef=0.2,p=1)],p=1)
RandomFog_img=RandomFog(image=Original)["image"]

# 51. Adding gravel 
RandomGravel=A.Compose([A.RandomGravel(gravel_roi=(0,0,1,1),number_of_patches=40,p=1)],p=1)
RandomGravel_img=RandomGravel(image=Original)["image"]

# 52. Adding tone curve 
RandomToneCurve=A.Compose([A.RandomToneCurve(scale=0.2,p=1)],p=1)
RandomToneCurve_img=RandomToneCurve(image=Original)["image"]

# 53. Frost
def apply_frost(image, **kwargs):
    alpha=0.5
    frost_texture = cv2.imread("frost/4.jpg")
    frost_texture = cv2.cvtColor(frost_texture, cv2.COLOR_BGR2RGB)
    frost_texture = cv2.resize(frost_texture, (image.shape[1],image.shape[0]), interpolation=cv2.INTER_LINEAR_EXACT)
    frosted_image = cv2.addWeighted(image, 1 - alpha, frost_texture, alpha, 0)
    return frosted_image

frost=A.Compose([A.Lambda(image=apply_frost)])
Frost_img=frost(image=Original)["image"]

In [ ]:
titles=['Original', 'AutoContrast', 'BoxBlur', 'CLAHE', 'ChannelShuffle', 'ChromaticAberration', 'CoarseDropout', 'ColorJitter', 
        'Defocus', 'Dilation', 'ElasticTransform', 'Emboss', 'Equalize', 'EqualizeYUV', 'Erosion', 'Frost', 'Gamma', 
        'GaussNoise', 'GaussianBlur', 'GlassBlur', 'HorizontalFlip', 'HueSaturationValue', 'Invert', 'JPEGCompression', 
        'MedianBlur', 'MotionBlur', 'OpticalDistortion', 'Perspective', 'Posterize', 'RGBShift', 'RandomBrightnessContrast', 
        'RandomFog', 'RandomGravel', 'RandomGridShuffle', 'RandomRain', 'RandomShadow', 'RandomSnow', 'RandomSunFlare', 
        'RandomToneCurve', 'Rotate', 'SaltandPepper', 'Scale', 'Sharpen', 'ShearX', 'ShearY', 'ShiftScaleRotate', 'Solarize', 
        'Speckle', 'ToGray', 'ToSepia', 'Translate', 'VerticalFlip', 'Vignette', 'ZoomBlur']

images=[Original, AutoContrast_img, BoxBlur_img, CLAHE_img, ChannelShuffle_img, ChromaticAberration_img, CoarseDropout_img, ColorJitter_img,
        Defocus_img, Dilation_img, ElasticTransform_img, Emboss_img, Equalize_img, EqualizeYUV_img, Erosion_img, Frost_img, Gamma_img,
        GaussNoise_img, GaussianBlur_img, GlassBlur_img, HorizontalFlip_img, HueSaturationValue_img, Invert_img, JPEGCompression_img,
        MedianBlur_img, MotionBlur_img, OpticalDistortion_img, Perspective_img, Posterize_img, RGBShift_img, RandomBrightnessContrast_img,
        RandomFog_img, RandomGravel_img, RandomGridShuffle_img, RandomRain_img, RandomShadow_img, RandomSnow_img, RandomSunFlare_img,
        RandomToneCurve_img, Rotate_img, SaltandPepper_img, Scale_img, Sharpen_img, ShearX_img, ShearY_img, ShiftScaleRotate_img,
        Solarize_img, Speckle_img, ToGray_img, ToSepia_img, Translate_img, VerticalFlip_img, Vignette_img, ZoomBlur_img]

In [ ]:
fig=plt.figure(figsize=(14,18))
fig.subplots_adjust(wspace=0.01, hspace=0.01)

for i in range(len(images)):
    plt.subplot(9,6,i+1)
    plt.imshow(images[i], cmap='gray')
    plt.title(titles[i])
    plt.axis('off')

plt.tight_layout()  
plt.show()

In [ ]:
print('Total_Augmented_Images: ', len(images)-1)

In [ ]:
path="Datasets/1.HAM10000/siamese/train/"
dest="Datasets/1.HAM10000/siamese/train_aug/"

folders=sorted(os.listdir(path))
for i in folders:   
    os.mkdir(dest+i)
    files=os.listdir(path+i)   
    for x in files:  
        image_name=x.split('.')[0]   
        image=cv2.imread(path+i+'/'+x)
        
        destination=dest+i+'/'+image_name+'/' 
        os.mkdir(destination)
        
        # 1. Original
        cv2.imwrite(destination+'{}'.format(image_name)+"_Original.png", image, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 2. Rotate
        Rotate_img=Rotate(image=image)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_Rotate.png", Rotate_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 3. Translate
        Translate_img=Translate(image=image)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_Translate.png", Translate_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 4. Scale
        Scale_img=Scale(image=image)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_Scale.png", Scale_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 5. ShiftScaleRotate
        ShiftScaleRotate_img=ShiftScaleRotate(image=image)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_ShiftScaleRotate.png", ShiftScaleRotate_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 6. HorizontalFlip
        HorizontalFlip_img=HorizontalFlip(image=image)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_HorizontalFlip.png", HorizontalFlip_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])

        # 7. VerticalFlip
        VerticalFlip_img=VerticalFlip(image=image)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_VerticalFlip.png", VerticalFlip_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])

        # 8. ShearX
        ShearX_img=ShearX(image=image)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_ShearX.png", ShearX_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 9. ShearY
        ShearY_img=ShearY(image=image)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_ShearY.png", ShearY_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 10. ColorJitter
        ColorJitter_img=ColorJitter(image=image)['image'] 
        cv2.imwrite(destination+'{}'.format(image_name)+"_ColorJitter.png", ColorJitter_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 11. RandomBrightnessContrast
        RandomBrightnessContrast_img=RandomBrightnessContrast(image=image)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_RandomBrightnessContrast.png", RandomBrightnessContrast_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 12. HueSaturationValue
        HueSaturationValue_img=HueSaturationValue(image=image)["image"]  
        cv2.imwrite(destination+'{}'.format(image_name)+"_HueSaturationValue.png", HueSaturationValue_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 13. Sharpen
        Sharpen_img=Sharpen(image=image)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_Sharpen.png", Sharpen_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 14. Gamma
        Gamma_img=Gamma(image=image)["image"] 
        cv2.imwrite(destination+'{}'.format(image_name)+"_Gamma.png", Gamma_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 15. InvertImg
        InvertImg_img=InvertImg(image=image)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_Invert.png", Invert_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 16. Posterize
        Posterize_img=Posterize(image=image)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_Posterize.png", Posterize_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 17. Solarize
        Solarize_img=Solarize(image=image)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_Solarize.png", Solarize_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 18. autocontrast
        auto_contrast_img = autocontrast(image, cutoff=3)
        cv2.imwrite(destination+'{}'.format(image_name)+"_AutoContrast.png", AutoContrast_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 19. ChromaticAberration
        ChromaticAberration_img=ChromaticAberration(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_ChromaticAberration.png", ChromaticAberration_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 20. RGBShift
        RGBShift_img=RGBShift(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_RGBShift.png", RGBShift_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 21. ToGray
        ToGray_img=ToGray(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_ToGray.png", ToGray_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 22. ToSepia
        ToSepia_img=ToSepia(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_ToSepia.png", ToSepia_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 23. GaussNoise
        GaussNoise_img=GaussNoise(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_GaussNoise.png", GaussNoise_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 24. GaussNoise
        salt_and_pepper_img=salt_and_pepper(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_SaltandPepper.png", SaltandPepper_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 25. speckle_noise
        speckle_noise_img=speckle_noise(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_Speckle.png", Speckle_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 26. ImageCompression
        ImageCompression_img=ImageCompression(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_JPEGCompression.png", JPEGCompression_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 27. Blur
        Blur_img=Blur(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_BoxBlur.png", BoxBlur_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 28. GaussianBlur
        GaussianBlur_img=GaussianBlur(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_GaussianBlur.png", GaussianBlur_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 29. MotionBlur
        MotionBlur_img=MotionBlur(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_MotionBlur.png", MotionBlur_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 30. MedianBlur
        MedianBlur_img=MedianBlur(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_MedianBlur.png", MedianBlur_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 31. GlassBlur
        GlassBlur_img=GlassBlur(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_GlassBlur.png", GlassBlur_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 32. ZoomBlur
        ZoomBlur_img=ZoomBlur(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_ZoomBlur.png", ZoomBlur_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 33. Defocus
        Defocus_img=Defocus(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_Defocus.png", Defocus_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 34. emboss
        emboss_img=emboss(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_Emboss.png", Emboss_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 35. ChannelShuffle
        ChannelShuffle_img=ChannelShuffle(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_ChannelShuffle.png", ChannelShuffle_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 36. CLAHE
        CLAHE_img=CLAHE(image=image)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_CLAHE.png", CLAHE_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 37. Equalize
        Equalize_img=Equalize(image=image)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_Equalize.png", Equalize_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 38. equalize_yuv
        yuv_equalized_img=equalize_yuv(image)
        cv2.imwrite(destination+'{}'.format(image_name)+"_EqualizeYUV.png", EqualizeYUV_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])        
        
        # 39. ElasticTransform
        ElasticTransform_img=ElasticTransform(image=image)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_ElasticTransform.png", ElasticTransform_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 40. OpticalDistortion
        OpticalDistortion_img=OpticalDistortion(image=image)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_OpticalDistortion.png", OpticalDistortion_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 41. Perspective
        Perspective_img=Perspective(image=image)['image']
        cv2.imwrite(destination+'{}'.format(image_name)+"_Perspective.png", Perspective_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 42. RandomGridShuffle
        RandomGridShuffle_img=RandomGridShuffle(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_RandomGridShuffle.png", RandomGridShuffle_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 43. CoarseDropout
        CoarseDropout_img=CoarseDropout(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_CoarseDropout.png", CoarseDropout_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])        
        
        # 44. dilation
        dilation_img=dilation(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_Dilation.png", Dilation_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 45. erosion
        erosion_img=erosion(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_Erosion.png", Erosion_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 46. vignette_trans
        vignette_img=vignette_trans(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_Vignette.png", Vignette_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 47. RandomShadow
        RandomShadow_img=RandomShadow(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_RandomShadow.png", RandomShadow_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 48. RandomRain
        RandomRain_img=RandomRain(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_RandomRain.png", RandomRain_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 49. RandomSnow
        RandomSnow_img=RandomSnow(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_RandomSnow.png", RandomSnow_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 50. RandomSunFlare
        RandomSunFlare_img=RandomSunFlare(image=image)["image"]    
        cv2.imwrite(destination+'{}'.format(image_name)+"_RandomSunFlare.png", RandomSunFlare_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 51. RandomFog
        RandomFog_img=RandomFog(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_RandomFog.png", RandomFog_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 52. RandomGravel
        RandomGravel_img=RandomGravel(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_RandomGravel.png", RandomGravel_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 53. RandomToneCurve
        RandomToneCurve_img=RandomToneCurve(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_RandomToneCurve.png", RandomToneCurve_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])
        
        # 54. Frost
        frost_img=frost(image=image)["image"]
        cv2.imwrite(destination+'{}'.format(image_name)+"_Frost.png", Frost_img, [cv2.IMWRITE_PNG_COMPRESSION, 0])

In [ ]:
def visualize(pairs, labels, to_show=6, num_col=3, predictions=None, test=False):   
    num_row = to_show // num_col if to_show // num_col != 0 else 1
    to_show = num_row * num_col

    fig, axes = plt.subplots(num_row, num_col, figsize=(5, 5))
    for i in range(to_show):
        if num_row == 1:
            ax = axes[i % num_col]
        else:
            ax = axes[i // num_col, i % num_col]

        ax.imshow(tf.concat([pairs[i][0], pairs[i][1]], axis=1))
        ax.set_axis_off()
        if test:
            ax.set_title("True: {} | Pred: {:.4f}".format(labels[i], predictions[i][0]))
        else:
            ax.set_title("Label: {}".format(labels[i]))
    if test:
        plt.tight_layout(rect=(0, 0, 1.9, 1.9), w_pad=0.0)
    else:
        plt.tight_layout(rect=(0, 0, 1.5, 1.5))
    plt.show()

visualize(pairs_train[:-1], labels_train[:-1], to_show=6, num_col=6)